# Import Required Libraries
Import the necessary libraries, including h5py and numpy.

In [1]:
# Import Required Libraries
import pandas as pd
import h5py
import numpy as np
import nibabel as nib
# h5py is used for handling HDF5 files
# numpy is used for numerical operations

# Load h5 File
Use h5py to load an h5 file from a specified path.

In [6]:
# Load h5 File
file_path = 'data/BraTS2020_training_data/content/data/volume_1_slice_0.h5'  # specify the path to your h5 file

# Open the h5 file in read mode
with h5py.File(file_path, 'r') as h5_file:
    # Explore the structure of the H5 file
    def print_structure(name, obj):
        print(name)
    h5_file.visititems(print_structure)
    
    # Access the 'image' dataset
    if 'image' in h5_file:
        image_data = h5_file['image']
        print("Image shape:", image_data.shape)
    else:
        print("Dataset 'image' not found in the file.")
    
    # Access the 'mask' dataset
    if 'mask' in h5_file:
        mask_data = h5_file['mask']
        print("Mask shape:", mask_data.shape)
    else:
        print("Dataset 'mask' not found in the file.")

image
mask
Image shape: (240, 240, 4)
Mask shape: (240, 240, 3)


In [2]:
import os
from PIL import Image
import numpy as np
# Directory containing the images
main_dir = 'data/NINS_Dataset/'
image_dir = 'Brain Atrophy'

# Initialize a list to store the images
image_list = []

# Iterate over all files in the directory
for filename in os.listdir(main_dir + image_dir):
    if filename.endswith('.png') or filename.endswith('.jpg'):  # Adjust the file extensions as needed
        image_path = os.path.join(main_dir + image_dir, filename)
        image = Image.open(image_path)
        image = image.resize((240, 240), Image.ANTIALIAS)
        image = image.convert('L')
        image_array = np.array(image)
        
        image_list.append(image_array)

# Stack images along a new dimension
stacked_images = np.stack(image_list, axis=0)

print("Stacked images shape:", stacked_images.shape)
out_path = 'data/NINS/' + image_dir + '.nii'
image_nii = nib.Nifti1Image(stacked_images, np.eye(4))
nib.save(image_nii, out_path)

Stacked images shape: (264, 240, 240)


In [ ]:

# Load the CSV file
csv_path = 'data/BraTS2020_training_data/content/data/meta_data.csv'
df = pd.read_csv(csv_path)

# Group by volume to process each volume separately
grouped = df.groupby('volume')

for volume_id, group in grouped:
    # Initialize lists to store image and mask slices
    image_slices = []
    mask_slices = []

    for _, row in group.iterrows():
        h5_path = "data/BraTS2020_training_data" + row['slice_path']
        
        with h5py.File(h5_path, 'r') as h5_file:
            # Assuming the datasets are named 'image' and 'mask'
            image_slices.append(h5_file['image'][:])
            mask_slices.append(h5_file['mask'][:])
    # Stack slices to form 3D volumes
    image_slices = np.array(image_slices)  # shape (num_slices, h, w, 4)
    mask_slices = np.array(mask_slices)    # shape (num_slices, h, w, 3)
    
    # Transpose to get shape (h, w, num_slices, 4) for images and (h, w, num_slices, 3) for masks
    image_volumes = np.transpose(image_slices, (1, 2, 0, 3))
    mask_volumes = np.transpose(mask_slices, (1, 2, 0, 3))
    
    # Save each MRI type and mask type as separate NIfTI images
    mri_types = ['type1', 'type2', 'type3', 'type4']
    mask_types = ['mask1', 'mask2', 'mask3']
    
    for i, mri_type in enumerate(mri_types):
        image_nii = nib.Nifti1Image(image_volumes[..., i], np.eye(4))
        image_nii_path = f'data/Brats/volume_{volume_id}_{mri_type}.nii.gz'
        nib.save(image_nii, image_nii_path)
        print(f'Saved {image_nii_path}')
    
    for j, mask_type in enumerate(mask_types):
        mask_nii = nib.Nifti1Image(mask_volumes[..., j], np.eye(4))
        mask_nii_path = f'data/Brats/volume_{volume_id}_{mask_type}.nii.gz'
        nib.save(mask_nii, mask_nii_path)
        print(f'Saved {mask_nii_path}')


# Explore h5 File Structure
Explore the structure of the h5 file, including groups and datasets.

In [ ]:
# Explore h5 File Structure

# Open the h5 file in read mode
with h5py.File(file_path, 'r') as h5_file:
    # Function to recursively explore the structure of the h5 file
    def explore_h5_structure(name, obj):
        if isinstance(obj, h5py.Group):
            print(f"Group: {name}")
        elif isinstance(obj, h5py.Dataset):
            print(f"Dataset: {name}, shape: {obj.shape}, dtype: {obj.dtype}")

    # Visit all items in the file
    h5_file.visititems(explore_h5_structure)

# Extract Data from h5 File
Extract specific datasets from the h5 file and convert them to numpy arrays for further analysis.

In [ ]:
# Extract Data from h5 File

# Open the h5 file in read mode
with h5py.File(file_path, 'r') as h5_file:
    # Extract specific datasets
    dataset_1 = h5_file['dataset_1'][:]  # replace 'dataset_1' with your actual dataset name
    dataset_2 = h5_file['dataset_2'][:]  # replace 'dataset_2' with your actual dataset name

# Convert datasets to numpy arrays
array_1 = np.array(dataset_1)
array_2 = np.array(dataset_2)

# Display the extracted data
print("Dataset 1:", array_1)
print("Dataset 2:", array_2)

In [2]:
import numpy as np

In [3]:
given_labels = np.load('data/labels_classes_priors/synthseg_segmentation_labels_2.0.npy')
label_names = np.load('data/labels_classes_priors/synthseg_segmentation_names_2.0.npy')
denoiser =  np.load('data/labels_classes_priors/synthseg_denoiser_labels_2.0.npy')
topological_classes = np.load('data/labels_classes_priors/synthseg_topological_classes_2.0.npy')
parcellation_labels = np.load('data/labels_classes_priors/synthseg_parcellation_labels.npy')
parcellation_names = np.load('data/labels_classes_priors/synthseg_parcellation_names.npy')
qc_labels = np.load('data/labels_classes_priors/synthseg_qc_labels.npy')
qc_names = np.load('data/labels_classes_priors/synthseg_qc_names.npy')

In [4]:

print( "denoiser", len(denoiser))
print("topological_classes", len(topological_classes))
print("given_labels", len(given_labels))
print("parcellation", len(parcellation_labels))
print("qc_labels", len(qc_labels))

denoiser 55
topological_classes 55
given_labels 55
parcellation 69
qc_labels 54


In [5]:
unique_num = 100
new_label = np.array([101,102,103])
new_names = np.array(['tumor-1', 'tumor-2', 'tumor-3'])
updated_label = np.append(given_labels, new_label)
updated_names = np.append(label_names, new_names)
new_denoiser = np.ones(len(new_label))
updated_denoiser = np.append(denoiser, new_denoiser)
new_topological_classes = np.ones(len(new_label))*unique_num
updated_topological_classes = np.append(topological_classes, new_topological_classes)

In [8]:
len(np.unique(updated_label))

36

In [36]:
import tensorflow as tf
import numpy as np
import h5py

In [40]:
import tensorflow as tf
from tensorflow.keras.models import load_model

In [41]:
model_path = "models/synthseg_2.0.h5"  # Replace with the actual path to your model

In [7]:
[None]*3 + [1]

[None, None, None, 1]

In [11]:
import os
import numpy as np
import tensorflow as tf
from ext.lab2im import layers
from ext.neuron import models as nrn_models

# Define the model path
model_path = "models/brats_synthseg_2.0.h5" # Replace with the actual path

# Define input parameters
input_shape = [None,None,None,1]  # 3D input with 1 channel
labels_segmentation = np.unique(updated_label)  # Example label list
n_levels = 5
nb_conv_per_level = 2
conv_size = 3
unet_feat_count = 24
feat_multiplier = 2
activation = 'elu'
sigma_smoothing = 0
flip_indices = None
gradients = False

# Define the build_model function
def build_model(path_model,
                input_shape,
                labels_segmentation,
                n_levels,
                nb_conv_per_level,
                conv_size,
                unet_feat_count,
                feat_multiplier,
                activation,
                sigma_smoothing,
                flip_indices,
                gradients):
    assert os.path.isfile(path_model), "The provided model path does not exist."

    # Get the number of labels
    n_labels_seg = len(labels_segmentation)

    # Build the UNet
    net = nrn_models.unet(input_shape=input_shape,
                          nb_labels=n_labels_seg,
                          nb_levels=n_levels,
                          nb_conv_per_level=nb_conv_per_level,
                          conv_size=conv_size,
                          nb_features=unet_feat_count,
                          feat_mult=feat_multiplier,
                          activation=activation,
                          batch_norm=-1)
    net.load_weights(path_model, by_name=True, skip_mismatch=True)

    # Smooth posteriors if specified
    if sigma_smoothing > 0:
        last_tensor = net.output
        last_tensor = layers.GaussianBlur(sigma=sigma_smoothing)(last_tensor)
        net = tf.keras.models.Model(inputs=net.inputs, outputs=last_tensor)

    return net

# Load the model
model = build_model(model_path,
                    input_shape,
                    labels_segmentation,
                    n_levels,
                    nb_conv_per_level,
                    conv_size,
                    unet_feat_count,
                    feat_multiplier,
                    activation,
                    sigma_smoothing,
                    flip_indices,
                    gradients)

# Print the model summary
model.summary()

Model: "unet"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
unet_input (InputLayer)         (None, None, None, N 0                                            
__________________________________________________________________________________________________
unet_conv_downarm_0_0 (Conv3D)  (None, None, None, N 672         unet_input[0][0]                 
__________________________________________________________________________________________________
unet_conv_downarm_0_1 (Conv3D)  (None, None, None, N 15576       unet_conv_downarm_0_0[0][0]      
__________________________________________________________________________________________________
unet_bn_down_0 (BatchNormalizat (None, None, None, N 96          unet_conv_downarm_0_1[0][0]      
_______________________________________________________________________________________________

In [50]:
model.output

<tf.Tensor 'unet_prediction_25/truediv:0' shape=(None, None, None, None, 33) dtype=float32>

In [53]:
model.output[0]

<tf.Tensor 'strided_slice:0' shape=(None, None, None, 33) dtype=float32>